# E11 — Selective Agent Evaluation (A2 GPT-RAG vs. A3 Selective Agent)

Reconstruction-v2. Real execution: 15 real `openai/gpt-5-mini` calls through the frozen E10 agent path; 135 non-triggered cases reuse A2 exactly (zero new hosted calls).

In [1]:
import json, os
from pathlib import Path
REPO = Path(os.environ.get('NDATRACE_REPO', Path.cwd()))
E11 = REPO / 'experiments/E11_selective_agent_evaluation'
E08B = REPO / 'experiments/E08B_stronger_model_diagnostic'
def load(p):
    with open(p) as f:
        return json.load(f)
print('repo:', REPO)

repo: /Users/asmitha/Documents/course/PE6201-EMERGING AI TECHNOLOGIES/Project/ndatrace


## 1. Research question

Does the frozen bounded selective agent improve evidence-grounded NDA review enough over frozen GPT-RAG to justify its added cost, latency, complexity, and failure surface?

## 2. Why E11 runs despite E09 no-go

E09 concluded A3 is not justified as the likely final architecture (0.67% of all 150 cases genuinely dynamic). E11 tests that conclusion empirically against a real, bounded prototype, for course completeness -- not to force an agent win.

## 3. Frozen A2/A3 setup

A2 = E08B's frozen GPT-RAG output (reused directly, not re-run). A3 = E10's frozen selective agent: trigger `cross_reference_to_named_provision_cue`, 2 tools (`follow_cross_reference`, `get_more_candidates`), hard limits unchanged, `openai/gpt-5-mini`.

## 4. Triggered population

In [2]:
wall = load(E11 / 'results/run_E11_wall_seconds.json')
print(json.dumps(wall, indent=2))

{
  "total_wall_seconds": 91.1206014159834,
  "n_triggered": 15,
  "run_id": "b8cca550fe90",
  "pre_run_ledger_usd": 0.3872392,
  "post_run_ledger_usd": 0.4090479499999999,
  "total_incremental_spend_usd": 0.021808749999999877
}


## 5. Execution integrity

In [3]:
rows = [json.loads(l) for l in open(E11 / 'results/run_E11_A3_train_cases.jsonl')]
print('n_rows:', len(rows))
assert len(rows) == 150
n_triggered = sum(1 for r in rows if r['triggered'])
print('n_triggered:', n_triggered)
assert n_triggered == 15

n_rows: 150
n_triggered: 15


## 6. A3 quality metrics

In [4]:
a3_summary = load(E11 / 'results/run_E11_A3_train.json')
print(json.dumps(a3_summary['classification'], indent=2, default=str))
print(json.dumps(a3_summary['evidence'], indent=2))
print(json.dumps(a3_summary['joint'], indent=2))

{
  "accuracy": 0.7933333333333333,
  "macro_f1": 0.7935946887346003,
  "per_class_recall": {
    "Entailment": 0.82,
    "Contradiction": 0.76,
    "NotMentioned": 0.8
  },
  "contradiction_recall": 0.76,
  "contradiction_recall_ci95": [
    0.6258705062480764,
    0.8570274759812778
  ],
  "contradiction_n": 50,
  "confusion_matrix": {
    "labels": [
      "Entailment",
      "Contradiction",
      "NotMentioned"
    ],
    "matrix": [
      [
        41,
        3,
        6
      ],
      [
        10,
        38,
        2
      ],
      [
        4,
        6,
        40
      ]
    ]
  }
}
{
  "evidence_bearing_n": 100,
  "evidence_recall": 0.84,
  "evidence_precision": 0.8235294117647058
}
{
  "overall": 0.74,
  "by_class": {
    "Entailment": 0.74,
    "Contradiction": 0.68,
    "NotMentioned": 0.8
  }
}


## 7. A2 vs A3 matched comparison

In [5]:
cmp = load(E11 / 'results/a2_vs_a3_paired_comparison.json')
import pandas as pd
pd.DataFrame(cmp['metric_table']).T

,A2,A3,delta_A3_minus_A2
accuracy,0.786667,0.793333,0.006667
macro_f1,0.786862,0.793595,0.006733
entailment_recall,0.820000,0.820000,0.000000
contradiction_recall,0.760000,0.760000,0.000000
notmentioned_recall,0.780000,0.800000,0.020000
evidence_recall,0.860000,0.840000,-0.020000
evidence_precision,0.834951,0.823529,-0.011422
joint_overall,0.740000,0.740000,0.000000
joint_entailment,0.740000,0.740000,0.000000
joint_contradiction,0.700000,0.680000,-0.020000


## 8. Classification transitions

In [6]:
print('Overall (150):', cmp['transitions_overall'])
print('Triggered-only (15):', cmp['transitions_triggered_only'])
print('Contradiction:', cmp['transitions_contradiction'])

Overall (150): {'a2_wrong_a3_correct': 1, 'a2_correct_a3_wrong': 0, 'both_correct': 118, 'both_wrong': 31}
Triggered-only (15): {'a2_wrong_a3_correct': 1, 'a2_correct_a3_wrong': 0, 'both_correct': 10, 'both_wrong': 4}
Contradiction: {'a2_wrong_a3_correct': 0, 'a2_correct_a3_wrong': 0, 'both_correct': 38, 'both_wrong': 12}


## 9. Joint transitions

In [7]:
print('Overall (150):', cmp['transitions_joint_overall'])
print('Triggered-only (15):', cmp['transitions_joint_triggered_only'])

Overall (150): {'a2_wrong_a3_correct': 1, 'a2_correct_a3_wrong': 1, 'both_correct': 110, 'both_wrong': 38}
Triggered-only (15): {'a2_wrong_a3_correct': 1, 'a2_correct_a3_wrong': 1, 'both_correct': 9, 'both_wrong': 4}


## 10. Agent behavior

In [8]:
print(json.dumps(a3_summary['agent_behavior'], indent=2, default=str))

{
  "n_triggered": 15,
  "escalation_rate": 0.1,
  "mean_steps": 1,
  "median_steps": 1,
  "mean_model_calls": 1,
  "mean_tool_calls": 0,
  "tool_usage_counts": {
    "FOLLOW_CROSS_REFERENCE": 0,
    "GET_MORE_CANDIDATES": 0
  },
  "cases_no_tool_before_final": 15,
  "fallback_count": 0,
  "fallback_rate": 0.0,
  "stop_reason_distribution": {
    "final": 15
  }
}


## 11. Tool usage

**Zero tool calls were made across all 15 triggered cases** -- every case concluded FINAL on step 1. `follow_cross_reference` and `get_more_candidates` were both available but never invoked.

## 12. Special case: train::273::nda-1 (E09's one confirmed dynamic case)

In [9]:
traces = {t['case_id']: t for t in (json.loads(l) for l in open(E11 / 'results/agent_traces.jsonl'))}
scored = {r['case_id']: r for r in (json.loads(l) for l in open(E11 / 'results/a3_scored_cases.jsonl'))}
cid = 'train::273::nda-1'
r, t = scored[cid], traces[cid]
print('gold:', r['gold_label'], 'a2:', r['a2_label'], 'a3:', r['a3_label'])
print('stop_reason:', t['stop_reason'], 'tools used:', t['tool_names'])
print('cost:', r['incremental_cost_usd'], 'latency_s:', r['incremental_latency_s'])
print('The agent did NOT invoke follow_cross_reference on this case despite being the exact '
      'case E09 identified as needing it -- it concluded FINAL immediately, unchanged from A2.')

gold: Entailment a2: NotMentioned a3: NotMentioned
stop_reason: final tools used: []
cost: 0.00100475 latency_s: 3.328045916976407
The agent did NOT invoke follow_cross_reference on this case despite being the exact case E09 identified as needing it -- it concluded FINAL immediately, unchanged from A2.


## 13. Special case: train::518::nda-10 (the only reachable retrieval-filtering case)

In [10]:
cid = 'train::518::nda-10'
r, t = scored[cid], traces[cid]
print('gold:', r['gold_label'], 'a2:', r['a2_label'], 'a3:', r['a3_label'])
print('stop_reason:', t['stop_reason'], 'tools used:', t['tool_names'])
print('get_more_candidates was NOT invoked -- the agent concluded FINAL immediately, '
      'unchanged from A2. This case remains unresolved under A3.')

gold: Entailment a2: NotMentioned a3: NotMentioned
stop_reason: final tools used: []
get_more_candidates was NOT invoked -- the agent concluded FINAL immediately, unchanged from A2. This case remains unresolved under A3.


## 14. Remaining 13 routed cases (false-positive-routing risk)

In [11]:
triggered_rows = [r for r in scored.values() if r['triggered']]
others = [r for r in triggered_rows if r['case_id'] not in ('train::273::nda-1','train::518::nda-10')]
import csv
a2_fail_rows = {rr['case_id']: rr for rr in csv.DictReader(open(E08B / 'results/gpt5mini_failure_analysis.csv'))}
already_correct = sum(1 for r in others if r['a2_label']==r['gold_label'])
already_joint = sum(1 for r in others if a2_fail_rows[r['case_id']]['joint_success']=='True')
unchanged = sum(1 for r in others if r['a2_label']==r['a3_label'])
improved = sum(1 for r in others if r['a2_label']!=r['gold_label'] and r['a3_label']==r['gold_label'])
regressed = sum(1 for r in others if r['a2_label']==r['gold_label'] and r['a3_label']!=r['gold_label'])
print(f'n={len(others)}, already A2-correct: {already_correct}, already A2-joint-success: {already_joint}')
print(f'unchanged label: {unchanged}, improved: {improved}, regressed: {regressed}')
print(f'avg incremental cost: ${sum(r["incremental_cost_usd"] for r in others)/len(others):.5f}')
print(f'avg incremental latency: {sum(r["incremental_latency_s"] for r in others)/len(others):.2f}s')

n=13, already A2-correct: 10, already A2-joint-success: 10
unchanged label: 12, improved: 1, regressed: 0
avg incremental cost: $0.00152
avg incremental latency: 6.42s


## 15. Safety/cap outcomes

In [12]:
print('fallback_count:', a3_summary['agent_behavior']['fallback_count'])
print('stop_reason_distribution:', a3_summary['agent_behavior']['stop_reason_distribution'])
print('No invalid actions, no tool errors, no cap hits occurred -- every case concluded FINAL cleanly on step 1.')

fallback_count: 0
stop_reason_distribution: {'final': 15}
No invalid actions, no tool errors, no cap hits occurred -- every case concluded FINAL cleanly on step 1.


## 16. Incremental cost (REAL, E11-only new spend)

In [13]:
print(json.dumps(a3_summary['incremental_cost_usd'], indent=2))
print(json.dumps(a3_summary['incremental_tokens'], indent=2))
print('total incremental spend:', wall['total_incremental_spend_usd'])

{
  "total": 0.02180875,
  "per_escalated_case": 0.0014539166666666665,
  "per_agent_call": 0.0014539166666666665
}
{
  "total_input": 22619,
  "total_output": 8077
}
total incremental spend: 0.021808749999999877


## 17. Blended deployment cost (separate from E11's own incremental spend)

In [14]:
a2_mean_cost = load(E08B / 'results/run_E08B_A2_gpt5mini_train.json')['cost_usd']['mean_per_case']
escalation_rate = a3_summary['agent_behavior']['escalation_rate']
incremental_per_escalated = a3_summary['incremental_cost_usd']['per_escalated_case']
blended_cost_per_case = a2_mean_cost + escalation_rate * incremental_per_escalated
print(f'A2 mean cost/case: ${a2_mean_cost:.5f}')
print(f'Blended A2+A3 cost/case: ${blended_cost_per_case:.5f}')
print(f'Projected 1,000-case cost: ${blended_cost_per_case*1000:.2f}')
print(f'Projected 8,000-case cost: ${blended_cost_per_case*8000:.2f}')
print(f'Percentage increase vs A2 alone: {(blended_cost_per_case/a2_mean_cost - 1)*100:.2f}%')

A2 mean cost/case: $0.00170
Blended A2+A3 cost/case: $0.00185
Projected 1,000-case cost: $1.85
Projected 8,000-case cost: $14.80
Percentage increase vs A2 alone: 8.53%


## 18. Latency

In [15]:
print('Incremental latency (triggered cases), forecast vs measured:')
print('  Forecast (pre-run):', load(E11/'results/pre_run_forecast.json')['runtime_forecast_seconds'])
print('  Measured:', a3_summary['incremental_latency_s'])

Incremental latency (triggered cases), forecast vs measured:
  Forecast (pre-run): {'minimum': 111.17148133304437, 'expected': 222.34296266608874, 'maximum': 564.7243706148583}
  Measured: {'mean': 6.074642247116814, 'median': 5.620381542015821, 'p90': 9.70008962508291, 'max': 9.780524958041497}


## 19. Paired statistics

In [16]:
print('McNemar overall:', cmp['mcnemar_overall'])
print('McNemar joint:', cmp['mcnemar_joint'])
print('McNemar Contradiction:', cmp['mcnemar_contradiction'])
print('Bootstrap accuracy delta:', cmp['bootstrap_accuracy_delta_A3_minus_A2'])
print('Bootstrap macro-F1 delta:', cmp['bootstrap_macro_f1_delta_A3_minus_A2'])
print('Bootstrap joint delta:', cmp['bootstrap_joint_delta_A3_minus_A2'])
print('Bootstrap Contradiction recall delta:', cmp['bootstrap_contradiction_recall_delta_A3_minus_A2'])
print('Only 1-2 discordant pairs possible with 15 triggered cases -- effect size/raw counts '
      'are the meaningful signal here, not the (necessarily non-significant) p-values.')

McNemar overall: {'b_a2_only_correct': 0, 'c_a3_only_correct': 1, 'n_discordant': 1, 'p_value': 1.0, 'significant_at_0.05': False}
McNemar joint: {'b_a2_only_correct': 1, 'c_a3_only_correct': 1, 'n_discordant': 2, 'p_value': 1.0, 'significant_at_0.05': False}
McNemar Contradiction: {'b_a2_only_correct': 0, 'c_a3_only_correct': 0, 'n_discordant': 0, 'p_value': 1.0, 'significant_at_0.05': False}
Bootstrap accuracy delta: {'point_estimate': 0.00666666666666671, 'ci95_low': 0.0, 'ci95_high': 0.020000000000000018}
Bootstrap macro-F1 delta: {'point_estimate': 0.00673258994319359, 'ci95_low': 0.0, 'ci95_high': 0.021973076428521865}
Bootstrap joint delta: {'point_estimate': 0.0, 'ci95_low': -0.020000000000000018, 'ci95_high': 0.020000000000000018}
Bootstrap Contradiction recall delta: {'point_estimate': 0.0, 'ci95_low': 0.0, 'ci95_high': 0.0}
Only 1-2 discordant pairs possible with 15 triggered cases -- effect size/raw counts are the meaningful signal here, not the (necessarily non-significant

## 20. Historical-agent warning

The T-series agent looked directionally favorable on a small dev sample (recovery beating regression ~2:1) but reversed at the full 2,091-case test set. E11's own real result here is consistent with caution about small-sample agent enthusiasm: zero tool calls were ever made, and the one net classification gain was exactly offset by one evidence-quality regression on an unrelated case -- net joint-success benefit is zero, not just "not yet significant."

## 21. Final A/B/C decision

**C — A3 CONFIRMS E09 NO-GO.**

The agent never invoked either available tool across all 15 triggered cases, including on the exact case (`train::273::nda-1`) E09 identified as needing cross-reference resolution, and on the one reachable retrieval-filtering case (`train::518::nda-10`). Net classification transitions: +1/-0 (a real but isolated gain). Net joint-success transitions: +1/-1 (a complete wash -- one case's evidence quality regressed even though its label didn't change). McNemar p=1.0 on both metrics (1-2 discordant pairs, as expected at this scale). Real incremental cost was incurred ($0.0218 total, ~$0.0015/escalated case) for zero net benefit. 

We implemented and evaluated a bounded selective read-only agent as a controlled architecture comparison. Agentic complexity was retained only if its measured benefit justified its additional cost, latency, and failure surface -- it did not, and this is a valid and useful course result, not a failure of the experiment.